# NeuroGolf 2026: Tiny-ONNX Solver
# Strategy: Ensemble multiple public notebooks + our own solver + dataset sources.
# Keep smallest valid model per task for maximum score.

In [ ]:
!pip install -q onnx onnxruntime onnx-tool 2>&1 | tail -1
import os, sys, json, math, re, zipfile, shutil, io, csv, copy, time, glob
from pathlib import Path
from collections import Counter
import numpy as np
import onnx
import onnx.helper as oh
import onnx.numpy_helper as onh
import onnxruntime as ort
from onnx import TensorProto
ort.set_default_logger_severity(3)

sys.path.insert(0, '/kaggle/input/competitions/neurogolf-2026/neurogolf_utils')
try:
    import neurogolf_utils as nu
    HAS_NU = True
    print('neurogolf_utils: OK')
except ImportError:
    HAS_NU = False
    print('[WARN] neurogolf_utils not found')

C, H, W = 10, 30, 30
HW = H * W
TASK_RE = re.compile(r'^task\d{3}\.onnx$')
MAX_BYTES = int(1.44 * 1024 * 1024)
EXCLUDED_OPS_UPPER = {'LOOP', 'SCAN', 'NONZERO', 'UNIQUE', 'SCRIPT', 'FUNCTION'}
NUM_TASKS = 400

COMP_DIR = Path('/kaggle/input/competitions/neurogolf-2026')
if not COMP_DIR.exists():
    COMP_DIR = Path('/kaggle/input/neurogolf-2026')
OUT_DIR = Path('/kaggle/working')
OUT_ZIP = OUT_DIR / 'submission.zip'

# Auto-discover ALL submission.zip files from kernel inputs
SOURCE_ZIPS = {}
for zp in Path('/kaggle/input').rglob('submission.zip'):
    label = zp.parent.name
    SOURCE_ZIPS[label] = zp

# Auto-discover ALL loose ONNX dirs
DATASET_DIRS = []
for d in Path('/kaggle/input').iterdir():
    if d.is_dir() and list(d.glob('task*.onnx'))[:1]:
        DATASET_DIRS.append(d)

print(f'Competition dir: {COMP_DIR}')
print(f'ZIP sources ({len(SOURCE_ZIPS)}): {list(SOURCE_ZIPS.keys())}')
print(f'ONNX dirs ({len(DATASET_DIRS)}): {[d.name for d in DATASET_DIRS]}')

In [ ]:
def grid_to_tensor(grid):
    g = np.array(grid, dtype=np.int32)
    h, w = g.shape
    t = np.zeros((1, C, H, W), dtype=np.float32)
    for r in range(h):
        for c in range(w):
            v = int(g[r, c])
            if 0 <= v < C: t[0, v, r, c] = 1.0
    return t

def verify_model(model, pairs):
    try:
        sess = ort.InferenceSession(model.SerializeToString(), providers=['CPUExecutionProvider'])
        for p in pairs:
            pred = sess.run(None, {'input': grid_to_tensor(p['input'])})[0]
            if not np.array_equal((pred > 0.0).astype(np.float32), grid_to_tensor(p['output'])):
                return False
        return True
    except Exception:
        return False

def estimate_cost(model):
    total = 0
    for init in model.graph.initializer:
        arr = onh.to_array(init)
        total += arr.size + arr.nbytes
    return max(total, 1)
print('Verification ready.')

## Detectors

In [ ]:
def same_size(pairs): return all(np.array(p['input']).shape == np.array(p['output']).shape for p in pairs)
def const_input_size(pairs): return len({np.array(p['input']).shape for p in pairs}) == 1

def detect_identity(pairs): return all(p['input'] == p['output'] for p in pairs)

def detect_color_map(pairs):
    if not same_size(pairs): return None
    cmap = {}
    for p in pairs:
        for a, b in zip(np.array(p['input']).flatten(), np.array(p['output']).flatten()):
            if int(a) in cmap and cmap[int(a)] != int(b): return None
            cmap[int(a)] = int(b)
    return cmap

def detect_rotation(pairs):
    for k in (1,2,3):
        ok = all(np.array_equal(np.rot90(np.array(p['input']), k), np.array(p['output'])) for p in pairs)
        if ok: return 90*k
    return None

def detect_flip(pairs):
    for axis, name in ((0,'vertical'),(1,'horizontal')):
        if all(np.array_equal(np.flip(np.array(p['input']), axis), np.array(p['output'])) for p in pairs):
            return name
    return None

def detect_transpose(pairs):
    for p in pairs:
        ig, og = np.array(p['input']), np.array(p['output'])
        if ig.T.shape != og.shape or not np.array_equal(ig.T, og): return False
    return True

def detect_tile(pairs):
    p0 = pairs[0]
    ih, iw = np.array(p0['input']).shape
    oh_, ow_ = np.array(p0['output']).shape
    if ih == 0 or iw == 0 or oh_ % ih or ow_ % iw: return None
    tr, tc = oh_//ih, ow_//iw
    for p in pairs:
        ig, og = np.array(p['input']), np.array(p['output'])
        if og.shape != (ig.shape[0]*tr, ig.shape[1]*tc): return None
        if not np.array_equal(np.tile(ig, (tr, tc)), og): return None
    return (tr, tc)

def detect_scale(pairs):
    for f in (2,3,4,5):
        ok = all(np.array_equal(
            np.repeat(np.repeat(np.array(p['input']), f, 0), f, 1),
            np.array(p['output'])) for p in pairs)
        if ok: return f
    return None

def detect_nonuniform_scale(pairs):
    for fh, fw in [(1,2),(2,1),(1,3),(3,1),(2,3),(3,2),(1,4),(4,1),(2,4),(4,2)]:
        ok = True
        for p in pairs:
            ig, og = np.array(p['input']), np.array(p['output'])
            if og.shape != (ig.shape[0]*fh, ig.shape[1]*fw): ok=False; break
            if not np.array_equal(np.repeat(np.repeat(ig, fh, 0), fw, 1), og): ok=False; break
        if ok: return (fh, fw)
    return None

def detect_mirror_h(pairs):
    for p in pairs:
        ig, og = np.array(p['input']), np.array(p['output'])
        if og.shape != (ig.shape[0], 2*ig.shape[1]): return False
        if not np.array_equal(np.concatenate([ig, np.flip(ig, 1)], 1), og): return False
    return True

def detect_mirror_v(pairs):
    for p in pairs:
        ig, og = np.array(p['input']), np.array(p['output'])
        if og.shape != (2*ig.shape[0], ig.shape[1]): return False
        if not np.array_equal(np.concatenate([ig, np.flip(ig, 0)], 0), og): return False
    return True

def detect_quad_mirror(pairs):
    for p in pairs:
        ig, og = np.array(p['input']), np.array(p['output'])
        if og.shape != (2*ig.shape[0], 2*ig.shape[1]): return False
        if not np.array_equal(np.block([
            [ig, np.flip(ig, 1)],
            [np.flip(ig, 0), np.flip(np.flip(ig, 0), 1)]
        ]), og): return False
    return True

def detect_shift(pairs):
    if not same_size(pairs): return None
    for dr in range(-5, 6):
        for dc in range(-5, 6):
            if dr == 0 and dc == 0: continue
            ok = True
            for p in pairs:
                ig, og = np.array(p['input']), np.array(p['output'])
                ih, iw = ig.shape
                shifted = np.zeros_like(ig)
                r0 = max(0, dr); r1 = min(ih, ih + dr); c0 = max(0, dc); c1 = min(iw, iw + dc)
                if r1 > r0 and c1 > c0:
                    sr0 = max(0, -dr); sc0 = max(0, -dc)
                    shifted[r0:r1, c0:c1] = ig[sr0:sr0+(r1-r0), sc0:sc0+(c1-c0)]
                if not np.array_equal(shifted, og): ok = False; break
            if ok: return (dr, dc)
    return None

def detect_fixed_crop(pairs):
    if not pairs or not const_input_size(pairs): return None
    p0 = pairs[0]
    ig0, og0 = np.array(p0['input']), np.array(p0['output'])
    ih, iw = ig0.shape; oh_, ow_ = og0.shape
    if oh_ > ih or ow_ > iw or (oh_ == ih and ow_ == iw): return None
    for r0 in range(ih - oh_ + 1):
        for c0 in range(iw - ow_ + 1):
            ok = True
            for p in pairs:
                ig, og = np.array(p['input']), np.array(p['output'])
                if og.shape != (oh_, ow_) or not np.array_equal(ig[r0:r0+oh_, c0:c0+ow_], og):
                    ok = False; break
            if ok: return (r0, c0, oh_, ow_)
    return None

def detect_rot_plus_color(pairs):
    for k in (1,2,3):
        cmap = {}; ok = True
        for p in pairs:
            ig, og = np.array(p['input']), np.array(p['output'])
            r = np.rot90(ig, k)
            if r.shape != og.shape: ok = False; break
            for a, b in zip(r.flatten(), og.flatten()):
                if int(a) in cmap and cmap[int(a)] != int(b): ok = False; break
                cmap[int(a)] = int(b)
            if not ok: break
        if ok and any(s != d for s, d in cmap.items()): return (90*k, cmap)
    return None

def detect_transpose_plus_color(pairs):
    cmap = {}
    for p in pairs:
        ig, og = np.array(p['input']), np.array(p['output'])
        if ig.T.shape != og.shape: return None
        for a, b in zip(ig.T.flatten(), og.flatten()):
            if int(a) in cmap and cmap[int(a)] != int(b): return None
            cmap[int(a)] = int(b)
    if any(s != d for s, d in cmap.items()): return cmap
    return None

def detect_flip_plus_color(pairs):
    for axis, name in ((0,'vertical'),(1,'horizontal')):
        cmap = {}; ok = True
        for p in pairs:
            ig, og = np.array(p['input']), np.array(p['output'])
            f = np.flip(ig, axis)
            if f.shape != og.shape: ok = False; break
            for a, b in zip(f.flatten(), og.flatten()):
                if int(a) in cmap and cmap[int(a)] != int(b): ok = False; break
                cmap[int(a)] = int(b)
            if not ok: break
        if ok and any(s != d for s, d in cmap.items()): return (name, cmap)
    return None

def _gravity(grid, direction):
    r = np.zeros_like(grid); h, w = grid.shape
    if direction in ('down', 'up'):
        for c in range(w):
            nz = grid[:, c][grid[:, c] != 0]
            if direction == 'down': r[h-len(nz):h, c] = nz
            else: r[:len(nz), c] = nz
    else:
        for rr in range(h):
            nz = grid[rr, :][grid[rr, :] != 0]
            if direction == 'right': r[rr, w-len(nz):w] = nz
            else: r[rr, :len(nz)] = nz
    return r

def detect_gravity(pairs):
    if not same_size(pairs): return None
    for d in ('down', 'up', 'left', 'right'):
        if all(np.array_equal(_gravity(np.array(p['input']), d), np.array(p['output'])) for p in pairs):
            return d
    return None

def detect_extract_outline(pairs):
    """Output = input with all interior cells set to 0."""
    if not same_size(pairs): return None
    for p in pairs:
        ig, og = np.array(p['input']), np.array(p['output'])
        h, w = ig.shape
        expected = np.zeros_like(ig)
        if h > 0 and w > 0:
            expected[0, :] = ig[0, :]
            expected[-1, :] = ig[-1, :]
            expected[:, 0] = ig[:, 0]
            expected[:, -1] = ig[:, -1]
        if not np.array_equal(expected, og): return False
    return True

print('Detectors ready.')

## Gather-index builders

In [ ]:
def rotation_gather(angle, ih, iw):
    k = angle // 90
    gi = np.zeros(H*W, dtype=np.int64)
    out_h, out_w = (ih, iw) if k == 2 else (iw, ih)
    for r in range(out_h):
        for c in range(out_w):
            if k == 1: sr, sc = c, iw-1-r
            elif k == 2: sr, sc = ih-1-r, iw-1-c
            else: sr, sc = ih-1-c, r
            if 0 <= sr < ih and 0 <= sc < iw: gi[r*W+c] = sr*W+sc
    return gi

def rotation_output_shape(angle, ih, iw):
    return (ih, iw) if (angle//90) == 2 else (iw, ih)

def flip_gather(direction, ih, iw):
    gi = np.zeros(H*W, dtype=np.int64)
    for r in range(ih):
        for c in range(iw):
            if direction == 'vertical': gi[r*W+c] = (ih-1-r)*W + c
            else: gi[r*W+c] = r*W + (iw-1-c)
    return gi

def transpose_gather(ih, iw):
    gi = np.zeros(H*W, dtype=np.int64)
    for r in range(iw):
        for c in range(ih):
            gi[r*W+c] = c*W + r
    return gi

def tile_gather(tr, tc, ih, iw):
    gi = np.zeros(H*W, dtype=np.int64)
    for r in range(min(H, ih*tr)):
        for c in range(min(W, iw*tc)):
            gi[r*W+c] = (r%ih)*W + (c%iw)
    return gi

def scale_gather(f, ih, iw):
    gi = np.zeros(H*W, dtype=np.int64)
    for r in range(min(H, ih*f)):
        for c in range(min(W, iw*f)):
            gi[r*W+c] = (r//f)*W + (c//f)
    return gi

def nonuniform_scale_gather(fh, fw, ih, iw):
    gi = np.zeros(H*W, dtype=np.int64)
    for r in range(min(H, ih*fh)):
        for c in range(min(W, iw*fw)):
            gi[r*W+c] = (r//fh)*W + (c//fw)
    return gi

def mirror_h_gather(ih, iw):
    gi = np.zeros(H*W, dtype=np.int64)
    for r in range(ih):
        for c in range(min(W, 2*iw)):
            sc = c if c < iw else 2*iw-1-c
            gi[r*W+c] = r*W + sc
    return gi

def mirror_v_gather(ih, iw):
    gi = np.zeros(H*W, dtype=np.int64)
    for r in range(min(H, 2*ih)):
        for c in range(iw):
            sr = r if r < ih else 2*ih-1-r
            gi[r*W+c] = sr*W + c
    return gi

def quad_mirror_gather(ih, iw):
    gi = np.zeros(H*W, dtype=np.int64)
    for r in range(min(H, 2*ih)):
        for c in range(min(W, 2*iw)):
            sr = r if r < ih else 2*ih-1-r
            sc = c if c < iw else 2*iw-1-c
            gi[r*W+c] = sr*W + sc
    return gi

def shift_gather(dr, dc, ih, iw):
    gi = np.zeros(H*W, dtype=np.int64)
    for r in range(ih):
        for c in range(iw):
            sr, sc = r-dr, c-dc
            if 0 <= sr < ih and 0 <= sc < iw:
                gi[r*W+c] = sr*W + sc
    return gi

def crop_gather(r0, c0, oh_, ow_):
    gi = np.zeros(H*W, dtype=np.int64)
    for r in range(oh_):
        for c in range(ow_):
            gi[r*W+c] = (r0+r)*W + (c0+c)
    return gi

def output_mask(oh_, ow_):
    m = np.zeros((1, C, H, W), dtype=np.float32)
    m[0, :, :oh_, :ow_] = 1.0
    return m

def build_color_weight(cmap):
    w = np.zeros((C, C, 1, 1), dtype=np.float32)
    for s, d in cmap.items():
        if 0 <= s < C and 0 <= d < C: w[d, s, 0, 0] = 1.0
    return w

def outline_conv_weight():
    """3x3 conv weights that keep a pixel iff at least one neighbor or self is out-of-grid (i.e., border)."""
    # This is a placeholder; we rely on the learned-conv fallback for the actual pattern.
    return None
print('Gather builders ready.')

## Learned-conv fallbacks (multi-seed + MSE/BCE + ternary snap)

In [ ]:
def _ternary_snap(w, eps=0.2):
    a = np.where(w > eps, 1.0, np.where(w < -eps, -1.0, 0.0)).astype(np.float32)
    return a

def try_learned_conv(train, all_pairs, kernel_size=1, steps=3000, lr=0.03, seeds=(0, 7, 42)):
    try:
        import torch, torch.nn as nn
    except ImportError:
        return None
    if not all(np.array(p['input']).shape == np.array(p['output']).shape for p in train):
        return None
    pad = kernel_size // 2
    inp = torch.tensor(np.stack([grid_to_tensor(p['input'])[0] for p in train]))
    out = torch.tensor(np.stack([grid_to_tensor(p['output'])[0] for p in train]))
    best_model = None
    best_cost = float('inf')
    for seed in seeds:
        torch.manual_seed(seed)
        conv = nn.Conv2d(C, C, kernel_size=kernel_size, padding=pad, bias=False)
        if seed == 0: nn.init.zeros_(conv.weight)
        opt = torch.optim.Adam(conv.parameters(), lr=lr)
        best_loss, best_state = float('inf'), None
        for _ in range(steps):
            opt.zero_grad()
            pred = conv(inp)
            loss = nn.functional.mse_loss(pred, out)
            loss.backward()
            opt.step()
            if loss.item() < best_loss:
                best_loss = loss.item()
                best_state = copy.deepcopy(conv.state_dict())
            if best_loss < 1e-8: break
        if best_state is None: continue
        conv.load_state_dict(best_state)
        w = conv.weight.detach().numpy()
        # Try the continuous weights first, then the ternary-snapped version.
        for w_cand, tag in [(w, 'float'), (_ternary_snap(w), 'ternary')]:
            m = make_conv_onnx(w_cand, kernel_size=kernel_size)
            if verify_model(m, all_pairs):
                c = estimate_cost(m)
                if c < best_cost:
                    best_cost, best_model = c, m
    return best_model

def try_two_layer_conv(train, all_pairs, ks1=3, ks2=1, hidden=C, steps=2500, lr=0.01, seeds=(0, 7)):
    try:
        import torch, torch.nn as nn
    except ImportError:
        return None
    if not all(np.array(p['input']).shape == np.array(p['output']).shape for p in train):
        return None
    inp = torch.tensor(np.stack([grid_to_tensor(p['input'])[0] for p in train]))
    out = torch.tensor(np.stack([grid_to_tensor(p['output'])[0] for p in train]))
    best_model = None; best_cost = float('inf')
    for seed in seeds:
        torch.manual_seed(seed)
        net = torch.nn.Sequential(
            torch.nn.Conv2d(C, hidden, kernel_size=ks1, padding=ks1//2, bias=False),
            torch.nn.ReLU(),
            torch.nn.Conv2d(hidden, C, kernel_size=ks2, padding=ks2//2, bias=False),
        )
        opt = torch.optim.Adam(net.parameters(), lr=lr)
        best_loss, best_state = float('inf'), None
        for _ in range(steps):
            opt.zero_grad()
            pred = net(inp)
            loss = torch.nn.functional.mse_loss(pred, out)
            loss.backward()
            opt.step()
            if loss.item() < best_loss:
                best_loss = loss.item()
                best_state = copy.deepcopy(net.state_dict())
            if best_loss < 1e-8: break
        if best_state is None: continue
        net.load_state_dict(best_state)
        w1 = net[0].weight.detach().numpy()
        w2 = net[2].weight.detach().numpy()
        for w1c, w2c in [(w1, w2), (_ternary_snap(w1), _ternary_snap(w2))]:
            m = make_two_layer_conv_onnx(w1c, w2c, ks1=ks1, ks2=ks2)
            if verify_model(m, all_pairs):
                c = estimate_cost(m)
                if c < best_cost: best_cost, best_model = c, m
    return best_model

print('Learned-conv fallbacks ready.')

## Main solver

In [ ]:
def solve_task(task):
    train = task.get('train', [])
    if not train: return None, None, None
    all_pairs = task_pairs(task)
    candidates = []

    def try_add(name, model):
        if model is None: return
        if verify_model(model, all_pairs):
            candidates.append((estimate_cost(model), name, model))

    const = const_input_size(all_pairs)
    ih, iw = np.array(train[0]['input']).shape

    if detect_identity(all_pairs):
        try_add('identity', make_identity_onnx())

    cmap = detect_color_map(train)
    if cmap is not None:
        try_add('color_map', make_conv1x1_onnx(build_color_weight(cmap)))

    if const:
        rot = detect_rotation(train)
        if rot is not None:
            oh_, ow_ = rotation_output_shape(rot, ih, iw)
            try_add(f'rot{rot}', make_gather_onnx(rotation_gather(rot, ih, iw), mask=output_mask(oh_, ow_)))

        fl = detect_flip(train)
        if fl is not None:
            try_add(f'flip_{fl}', make_gather_onnx(flip_gather(fl, ih, iw), mask=output_mask(ih, iw)))

        if detect_transpose(train):
            try_add('transpose', make_gather_onnx(transpose_gather(ih, iw), mask=output_mask(iw, ih)))

        t = detect_tile(train)
        if t:
            tr, tc = t
            try_add(f'tile_{tr}x{tc}', make_gather_onnx(tile_gather(tr, tc, ih, iw),
                                                       mask=output_mask(min(H, ih*tr), min(W, iw*tc))))

        s = detect_scale(train)
        if s:
            try_add(f'scale_{s}', make_gather_onnx(scale_gather(s, ih, iw),
                                                  mask=output_mask(min(H, ih*s), min(W, iw*s))))

        nus = detect_nonuniform_scale(train)
        if nus:
            fh, fw = nus
            try_add(f'scale_{fh}x{fw}', make_gather_onnx(nonuniform_scale_gather(fh, fw, ih, iw),
                                                        mask=output_mask(min(H, ih*fh), min(W, iw*fw))))

        if detect_mirror_h(train):
            try_add('mirror_h', make_gather_onnx(mirror_h_gather(ih, iw), mask=output_mask(ih, min(W, 2*iw))))
        if detect_mirror_v(train):
            try_add('mirror_v', make_gather_onnx(mirror_v_gather(ih, iw), mask=output_mask(min(H, 2*ih), iw)))
        if detect_quad_mirror(train):
            try_add('quad_mirror', make_gather_onnx(quad_mirror_gather(ih, iw),
                                                   mask=output_mask(min(H, 2*ih), min(W, 2*iw))))

        sh = detect_shift(train)
        if sh is not None:
            dr, dc = sh
            try_add(f'shift_{dr}_{dc}', make_gather_onnx(shift_gather(dr, dc, ih, iw), mask=output_mask(ih, iw)))

        fc = detect_fixed_crop(train + task.get('test', []) + task.get('arc-gen', [])[:3])
        if fc is not None:
            r0, c0, oh_, ow_ = fc
            try_add(f'crop_{r0}_{c0}', make_gather_onnx(crop_gather(r0, c0, oh_, ow_),
                                                       mask=output_mask(oh_, ow_)))

        rc = detect_rot_plus_color(train)
        if rc is not None:
            ang, cmap_ = rc
            oh_, ow_ = rotation_output_shape(ang, ih, iw)
            try_add(f'rot{ang}+color', make_gather_then_conv1x1_onnx(
                rotation_gather(ang, ih, iw), build_color_weight(cmap_), mask=output_mask(oh_, ow_)))

        fc2 = detect_flip_plus_color(train)
        if fc2 is not None:
            name, cmap_ = fc2
            try_add(f'flip_{name}+color', make_gather_then_conv1x1_onnx(
                flip_gather(name, ih, iw), build_color_weight(cmap_), mask=output_mask(ih, iw)))

        tpc = detect_transpose_plus_color(train)
        if tpc is not None:
            try_add('transpose+color', make_gather_then_conv1x1_onnx(
                transpose_gather(ih, iw), build_color_weight(tpc), mask=output_mask(iw, ih)))

    # Single-layer learned conv: tries k=1, 3, 5 with multiple seeds; picks smallest.
    for ks in (1, 3, 5):
        m = try_learned_conv(train, all_pairs, kernel_size=ks, steps=3000)
        if m is not None:
            try_add(f'learned_conv_{ks}', m)

    # Two-layer conv with ReLU for patterns that need non-linearity (edge/outline/erode).
    for ks1 in (3, 5):
        m = try_two_layer_conv(train, all_pairs, ks1=ks1, ks2=1)
        if m is not None:
            try_add(f'two_layer_{ks1}_1', m)

    if not candidates: return None, None, None
    candidates.sort(key=lambda c: c[0])
    return candidates[0][2], candidates[0][1], candidates[0][0]
print('Solver ready.')

## Blend Pipeline: ZIP sources + dataset + automated solver

In [ ]:
import shutil

# Load task JSONs
task_jsons = {}
for jp in COMP_DIR.rglob('*.json'):
    key = jp.stem + '.onnx'
    if TASK_RE.match(key):
        try: task_jsons[key] = json.loads(jp.read_text())
        except: pass
print(f'Task JSONs: {len(task_jsons)}')

def encode_grid(grid):
    t = np.zeros((1, C, H, W), dtype=np.float32)
    for r, row in enumerate(grid):
        if r >= H: break
        for c, v in enumerate(row):
            if c >= W: break
            if 0 <= v < C: t[0, v, r, c] = 1.0
    return t

def validate_raw(raw, pairs):
    try:
        sess = ort.InferenceSession(raw, providers=['CPUExecutionProvider'])
        inp_name = sess.get_inputs()[0].name
        for p in pairs:
            out = sess.run(None, {inp_name: encode_grid(p['input'])})[0]
            tgt = np.array(p['output']); th, tw = tgt.shape
            pred = np.argmax(out[0, :, :th, :tw], axis=0)
            if not np.array_equal(pred, tgt): return False
        return True
    except: return False

def strict_validate(raw, task_id):
    """Use official neurogolf_utils to fully validate + score a model."""
    if not HAS_NU: return None
    if len(raw) > MAX_BYTES: return None
    tmp = f'/tmp/_v{task_id:03d}.onnx'
    try:
        with open(tmp, 'wb') as f: f.write(raw)
        if not nu.check_network(tmp): return None
        sess = ort.InferenceSession(raw, providers=['CPUExecutionProvider'])
        examples = nu.load_examples(task_id)
        agi_r, agi_f, _ = nu.verify_subset(sess, examples['train'] + examples['test'])
        gen_r, gen_f, _ = nu.verify_subset(sess, examples['arc-gen'])
        if agi_f > 0 or gen_f > 0: return None
        macs, mem, params = nu.score_network(tmp)
        if None in (macs, mem, params): return None
        cost = int(macs + mem + params)
        return cost
    except:
        return None
    finally:
        if os.path.exists(tmp): os.remove(tmp)

def profile_raw(raw):
    if len(raw) > MAX_BYTES: return False, float('inf')
    try: model = onnx.load_from_string(raw)
    except: return False, float('inf')
    ops = {nd.op_type.upper() for nd in model.graph.node}
    if EXCLUDED_OPS_UPPER & ops: return False, float('inf')
    params = nbytes = macs = 0
    tensors = {}
    for init in model.graph.initializer:
        a = onh.to_array(init); tensors[init.name] = a
        params += a.size; nbytes += a.nbytes
    for nd in model.graph.node:
        if nd.op_type == 'Constant':
            for attr in nd.attribute:
                if attr.t:
                    try:
                        a = onh.to_array(attr.t)
                        if nd.output: tensors[nd.output[0]] = a
                        params += a.size; nbytes += a.nbytes
                    except: pass
    for nd in model.graph.node:
        if nd.op_type == 'Conv' and len(nd.input) >= 2 and nd.input[1] in tensors:
            w = tensors[nd.input[1]]
            if w.ndim == 4: macs += w.shape[0]*w.shape[1]*w.shape[2]*w.shape[3]*H*W
    return True, int(params + nbytes + macs)

# ═══ Phase 1: ZIP Blend ═══
print('=== Phase 1: ZIP Blend ===')
best = {}
costs = {}
src_ct = Counter()

for label, zpath in SOURCE_ZIPS.items():
    count = 0
    with zipfile.ZipFile(zpath, 'r') as zf:
        for entry in zf.namelist():
            base = os.path.basename(entry)
            if not TASK_RE.match(base): continue
            task_id = int(base[4:7])
            raw = zf.read(entry)
            ok, proxy_cost = profile_raw(raw)
            if not ok: continue
            # Strict validate with official utils
            real_cost = strict_validate(raw, task_id)
            if real_cost is None: continue
            if base not in best or real_cost < costs[base]:
                best[base] = raw; costs[base] = real_cost; count += 1
                src_ct[f'zip_{label}'] += 1
    print(f'  [{label}] {count} valid models')
print(f'After ZIP blend: {len(best)} tasks')

# ═══ Phase 2: Dataset ONNX ═══
print('\n=== Phase 2: Dataset ONNX ===')
for pdir in DATASET_DIRS:
    if not pdir.exists(): continue
    count = 0
    for f in sorted(pdir.glob('task*.onnx')):
        key = f.name
        if not TASK_RE.match(key): continue
        task_id = int(key[4:7])
        raw = f.read_bytes()
        if len(raw) == 0: continue
        ok, proxy_cost = profile_raw(raw)
        if not ok: continue
        real_cost = strict_validate(raw, task_id)
        if real_cost is None: continue
        if key not in best or real_cost < costs[key]:
            best[key] = raw; costs[key] = real_cost; count += 1
            src_ct['dataset'] += 1
    print(f'  {pdir.name}: +{count}')
print(f'After datasets: {len(best)} tasks')

# ═══ Phase 3: Automated solver ═══
print('\n=== Phase 3: Automated solver ===')
task_files = sorted(COMP_DIR.glob('task*.json'))
unsolved = [fp for fp in task_files if fp.stem + '.onnx' not in best]
print(f'Remaining: {len(unsolved)}')
nano_ok = 0
for fp in unsolved:
    key = fp.stem + '.onnx'
    task_id = int(fp.stem.replace('task', ''))
    try:
        task = json.loads(fp.read_text())
        model, name, _ = solve_task(task)
    except: continue
    if model is None: continue
    raw = model.SerializeToString()
    real_cost = strict_validate(raw, task_id)
    if real_cost is None: continue
    if key not in best or real_cost < costs.get(key, float('inf')):
        best[key] = raw; costs[key] = real_cost
        src_ct[f'solver_{name}'] += 1; nano_ok += 1
print(f'Solver: +{nano_ok}')
print(f'TOTAL: {len(best)}/{NUM_TASKS}')

# ═══ Budget check ═══
buf = io.BytesIO()
with zipfile.ZipFile(buf, 'w', zipfile.ZIP_DEFLATED) as zf:
    for name in sorted(best): zf.writestr(name, best[name])
sz = buf.tell()
print(f'\nZIP size: {sz/1024:.1f} KB / {MAX_BYTES/1024:.0f} KB limit')
if sz > MAX_BYTES:
    print('Over budget! Dropping most expensive...')
    by_cost = sorted(costs.items(), key=lambda x: -x[1])
    for tk, c in by_cost:
        if tk in best:
            del best[tk]; del costs[tk]
            buf2 = io.BytesIO()
            with zipfile.ZipFile(buf2, 'w', zipfile.ZIP_DEFLATED) as zf:
                for name in sorted(best): zf.writestr(name, best[name])
            if buf2.tell() <= MAX_BYTES:
                print(f'  Dropped to {len(best)} tasks, {buf2.tell()/1024:.1f} KB')
                break

# ═══ Write ═══
with zipfile.ZipFile(OUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for name in sorted(best): zf.writestr(name, best[name])

est_score = sum(max(1.0, 25.0 - math.log(c)) for c in costs.values())
print(f'\n=== FINAL ===')
print(f'Tasks: {len(best)}/{NUM_TASKS}')
print(f'Est score: {est_score:.1f}')
print(f'ZIP: {OUT_ZIP.stat().st_size/1024:.1f} KB')
for lb, cnt in sorted(src_ct.items(), key=lambda x: -x[1]):
    print(f'  {lb}: {cnt}')